# grad-tracking-global-toggle — faded example 2: Faded: no_grad decorator uses global keyword to mutate flag

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `grad-tracking-global-toggle`. The last cell reports your progress on the `Backprop: Grad-tracking toggle` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad-tracking toggle` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-tracking-global-toggle`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-tracking-global-toggle"
DD_SUBTOPIC = "Backprop: Grad-tracking toggle"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When implementing a `no_grad` decorator, the wrapper must use the `global` keyword before assigning to `grad_tracking_enabled`. Without it, Python's scoping rules treat any assignment inside the function as creating a new local variable, leaving the module-level flag unchanged — a silent bug where the decorator appears to work but never actually disables tracking.

## Faded exercise 2

Complete the wrapper inside `no_grad`. The blank is the three-line body that: (1) declares the global, (2) snapshots the flag into `prev`, and (3) sets the flag to `False`. The `try/finally` restore is already provided.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
grad_tracking_enabled = True

def no_grad(fn):
    def wrapper(*args, **kwargs):
        global grad_tracking_enabled
        prev = grad_tracking_enabled
        grad_tracking_enabled = False
        try:
            return fn(*args, **kwargs)
        finally:
            grad_tracking_enabled = prev
    return wrapper

@no_grad
def my_fn():
    return globals()['grad_tracking_enabled']

print(my_fn())  # should print False
print(globals()['grad_tracking_enabled'])  # should be True (restored)


def _test():
    # Flag starts True
    assert globals()['grad_tracking_enabled'] == True

    # Inside the decorated function, flag is False
    @no_grad
    def check_inside():
        return globals()['grad_tracking_enabled']

    assert check_inside() == False, "Flag must be False inside no_grad-decorated function"
    assert globals()['grad_tracking_enabled'] == True, "Flag must be restored after decorated fn returns"

    # Exception safety
    @no_grad
    def crashing():
        raise RuntimeError("boom")

    try:
        crashing()
    except RuntimeError:
        pass
    assert globals()['grad_tracking_enabled'] == True, "Flag must be restored even after exception"


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
grad_tracking_enabled = True

def no_grad(fn):
    def wrapper(*args, **kwargs):
        global grad_tracking_enabled
        prev = grad_tracking_enabled
        grad_tracking_enabled = False
        try:
            return fn(*args, **kwargs)
        finally:
            grad_tracking_enabled = prev
    return wrapper

@no_grad
def my_fn():
    return globals()['grad_tracking_enabled']

print(my_fn())  # should print False
print(globals()['grad_tracking_enabled'])  # should be True (restored)
```
</details>